# Track C — 02. Fusions Cleaning

Three-stage collapse of the per-breakpoint DepMap fusion table to one row per `(ensg_id, model_id)`:
1. Filter to default sequencing entry, resolve gene IDs, dedupe breakpoint multiplicity → one row per fusion event
2. Explode each event into its two gene participants, collapse to gene-model grain
3. Validate every `ensg_id` (join key) against the protein-coding gene universe

Writes to `Track - C/outputs/fusions/`.

In [ ]:
import os
import numpy as np
import pandas as pd
from data_utils import PARQUET_CLEAN, REF

PROJECT_ROOT = os.path.abspath(os.path.join(REF, '..'))
OUT_DIR      = os.path.join(PROJECT_ROOT, 'Track - C', 'outputs', 'fusions')
os.makedirs(OUT_DIR, exist_ok=True)

CONF_RANK = {'high': 3, 'medium': 2, 'low': 1}
print('output dir:', OUT_DIR)


## 1. Event-level dedup + identifier resolution

In [ ]:
df = pd.read_parquet(os.path.join(PARQUET_CLEAN, 'fusions_clean.parquet'))

df = df[df['isdefaultentryformodel'] == 'yes'].copy()
print(f'After default-entry filter: {len(df)} rows, {df["modelid"].nunique()} models')

df['modelid'] = df['modelid'].str.upper()
df['gene1_ens_id'] = df['gene1_ens_id'].replace('.', np.nan).str.upper()
df['gene2_ens_id'] = df['gene2_ens_id'].replace('.', np.nan).str.upper()

gene_lookup = pd.read_parquet(os.path.join(REF, 'gene_lookup.parquet'))
symbol_to_ensg = gene_lookup.set_index(gene_lookup['hgnc_symbol'].str.upper())['ensg_id'].to_dict()

def resolve_ensg(row, ens_col, symbol_col):
    if pd.notna(row[ens_col]):
        return row[ens_col]
    return symbol_to_ensg.get(str(row[symbol_col]).upper(), np.nan)

df['gene1_ens_id_resolved'] = df.apply(lambda r: resolve_ensg(r, 'gene1_ens_id', 'gene1'), axis=1)
df['gene2_ens_id_resolved'] = df.apply(lambda r: resolve_ensg(r, 'gene2_ens_id', 'gene2'), axis=1)

unmapped_g1 = df[df['gene1_ens_id_resolved'].isna()][['modelid', 'canonicalfusionname', 'gene1']]
unmapped_g2 = df[df['gene2_ens_id_resolved'].isna()][['modelid', 'canonicalfusionname', 'gene2']]
unmapped_g1.to_csv(os.path.join(OUT_DIR, 'fusions_unmapped_gene1.csv'), index=False)
unmapped_g2.to_csv(os.path.join(OUT_DIR, 'fusions_unmapped_gene2.csv'), index=False)
print(f'Unmapped gene1: {len(unmapped_g1)} ({len(unmapped_g1)/len(df):.2%})')
print(f'Unmapped gene2: {len(unmapped_g2)} ({len(unmapped_g2)/len(df):.2%})')

df['confidence_rank'] = df['confidence'].map(CONF_RANK)

df_sorted = df.sort_values(['modelid', 'canonicalfusionname', 'ffpm', 'confidence_rank'],
                            ascending=[True, True, False, False])
events = df_sorted.drop_duplicates(subset=['modelid', 'canonicalfusionname'], keep='first').copy()
print(f'Before event dedup: {len(df)} breakpoint calls')
print(f'After event dedup:  {len(events)} fusion events')


## 2. Explode to gene level, collapse to (ensg_id, model_id)

In [ ]:
side1 = events.rename(columns={
    'gene1_ens_id_resolved': 'ensg_id', 'gene1': 'gene_symbol',
    'gene2_ens_id_resolved': 'partner_ensg_id', 'gene2': 'partner_gene_symbol'
})[['modelid', 'ensg_id', 'gene_symbol', 'partner_ensg_id', 'partner_gene_symbol',
    'ffpm', 'confidence', 'confidence_rank', 'reading_frame', 'canonicalfusionname']]
side2 = events.rename(columns={
    'gene2_ens_id_resolved': 'ensg_id', 'gene2': 'gene_symbol',
    'gene1_ens_id_resolved': 'partner_ensg_id', 'gene1': 'partner_gene_symbol'
})[['modelid', 'ensg_id', 'gene_symbol', 'partner_ensg_id', 'partner_gene_symbol',
    'ffpm', 'confidence', 'confidence_rank', 'reading_frame', 'canonicalfusionname']]
exploded = pd.concat([side1, side2], ignore_index=True)
exploded = exploded.dropna(subset=['ensg_id'])
print(f'Exploded rows: {len(exploded)}')

exploded['is_in_frame'] = exploded['reading_frame'] == 'in-frame'

def collapse_group(g):
    best = g.loc[g['ffpm'].idxmax()]
    return pd.Series({
        'fusion_count': g['canonicalfusionname'].nunique(),
        'max_confidence': g.loc[g['confidence_rank'].idxmax(), 'confidence'],
        'best_ffpm': g['ffpm'].max(),
        'any_in_frame': g['is_in_frame'].any(),
        'top_partner_ensg_id': best['partner_ensg_id'],
        'top_partner_gene': best['partner_gene_symbol'],
    })

fusions_clean = exploded.groupby(['ensg_id', 'modelid']).apply(collapse_group, include_groups=False).reset_index()
fusions_clean = fusions_clean.rename(columns={'modelid': 'model_id'})
print(f'Final grain: {len(fusions_clean)} rows, {fusions_clean["ensg_id"].nunique()} genes, {fusions_clean["model_id"].nunique()} models')


In [ ]:
# Strip Ensembl version suffixes after explosion, then re-collapse to gene-model grain.
# The groupby in the cell above already ran on unstripped IDs, so we redo it here.
# (e.g. ENSG00000139618.10 -> ENSG00000139618)
exploded['ensg_id'] = exploded['ensg_id'].str.split('.').str[0]
print(f'After version strip: {exploded["ensg_id"].nunique():,} unique ENSG IDs in exploded')

fusions_clean = exploded.groupby(['ensg_id', 'modelid']).apply(collapse_group, include_groups=False).reset_index()
fusions_clean = fusions_clean.rename(columns={'modelid': 'model_id'})
print(f'Re-collapsed grain: {len(fusions_clean)} rows, {fusions_clean["ensg_id"].nunique()} genes, {fusions_clean["model_id"].nunique()} models')

## 3. Restrict to gene universe

Only `ensg_id` (the join key) is validated against `gene_lookup`. `top_partner_ensg_id` /
`top_partner_gene` are metadata for the explanation layer, not a join key — a partner
outside the gene universe is legitimate signal and is left as-is.

In [ ]:
valid_ensg = set(gene_lookup['ensg_id'])
print(f'Before universe filter: {len(fusions_clean)} rows, {fusions_clean["ensg_id"].nunique()} unique genes')

out_of_universe = fusions_clean[~fusions_clean['ensg_id'].isin(valid_ensg)]
print(f'Out-of-universe rows: {len(out_of_universe)} ({len(out_of_universe)/len(fusions_clean):.2%})')
out_of_universe.to_csv(os.path.join(OUT_DIR, 'fusions_out_of_universe.csv'), index=False)

fusions_in_universe = fusions_clean[fusions_clean['ensg_id'].isin(valid_ensg)].copy()
print(f'After universe filter: {len(fusions_in_universe)} rows, {fusions_in_universe["ensg_id"].nunique()} unique genes')

OUT_PATH = os.path.join(OUT_DIR, 'fusions_gene_level.parquet')
fusions_in_universe.to_parquet(OUT_PATH)
print(f'\nSaved to {OUT_PATH}')
